In [1]:
import ast
import os
import pickle

import numpy as np
import pandas as pd
from yaml import safe_load
from tqdm import tqdm

c:\Users\Prakshil\Anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
csv_path = 'df.csv'
required_cols = [
    'innings',
    'info.teams',
    'info.city',
    'info.venue',
    'info.gender',
    'info.match_type',
    'info.overs'
 ]

def _use_col(col_name):
    return any(
        col_name == base or col_name.startswith(f'{base}.')
        for base in required_cols
    )

raw_df = pd.read_csv(
    csv_path,
    usecols=_use_col,
    memory_map=True
 )

def _resolve_column(df, base_name):
    if base_name in df.columns:
        return base_name
    candidates = [col for col in df.columns if col.startswith(f'{base_name}.')]
    return candidates[0] if candidates else None

def _safe_literal_eval(value):
    if isinstance(value, str):
        value = value.strip()
        if value and value[0] in '[{':
            try:
                return ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return value
    return value

matches = raw_df.copy()

missing_required = []
for col in required_cols:
    resolved = _resolve_column(matches, col)
    if resolved is None:
        missing_required.append(col)
    elif resolved != col:
        matches[col] = matches[resolved]

for col in ['innings', 'info.teams']:
    if col in matches.columns:
        matches[col] = matches[col].apply(_safe_literal_eval)

print(f'Loaded {csv_path} with shape: {matches.shape}')
print('Missing required columns:', missing_required if missing_required else 'None')
print('Columns in use:', matches.columns.tolist())

Loaded df.csv with shape: (4981, 7)
Missing required columns: None
Columns in use: ['innings', 'info.city', 'info.gender', 'info.match_type', 'info.overs', 'info.teams', 'info.venue']


In [3]:
backup = matches.copy()
matches.to_csv('matches_backup.csv', index=False)
matches


,innings,info.city,info.gender,info.match_type,info.overs,info.teams,info.venue
0,"[{'1st innings': {'team': 'England', 'deliveri...",Bristol,male,T20,20,"[England, India]",County Ground
1,"[{'1st innings': {'team': 'India', 'deliveries...",Delhi,male,T20,20,"[India, New Zealand]",Arun Jaitley Stadium
2,"[{'1st innings': {'team': 'New Zealand', 'deli...",Rajkot,male,T20,20,"[New Zealand, India]",Saurashtra Cricket Association Stadium
3,"[{'1st innings': {'team': 'India', 'deliveries...",Thiruvananthapuram,male,T20,20,"[India, New Zealand]",Greenfield International Stadium
4,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",Abu Dhabi,male,T20,20,"[Sri Lanka, Pakistan]",Sheikh Zayed Stadium
...,...,...,...,...,...,...,...
4976,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",Mirpur,male,T20,20,"[Pakistan, Sri Lanka]",Shere Bangla National Stadium
4977,"[{'1st innings': {'team': 'Bangladesh', 'deliv...",Mirpur,male,T20,20,"[Bangladesh, India]",Shere Bangla National Stadium
4978,"[{'1st innings': {'team': 'Netherlands', 'deli...",Dubai,male,T20,20,"[United Arab Emirates, Netherlands]",ICC Academy
4979,"[{'1st innings': {'team': 'Australia', 'delive...",NaN,male,T20,20,"[Sri Lanka, Australia]",Pallekele International Cricket Stadium


In [4]:
matches[['innings', 'info.teams', 'info.city', 'info.venue']].head(2)

,innings,info.teams,info.city,info.venue
0,"[{'1st innings': {'team': 'England', 'deliveri...","[England, India]",Bristol,County Ground
1,"[{'1st innings': {'team': 'India', 'deliveries...","[India, New Zealand]",Delhi,Arun Jaitley Stadium


In [5]:
matches = matches.drop(columns=[
    'meta.data_version',
    'meta.created',
    'meta.revision',
    'info.outcome.bowl_out',
    'info.bowl_out',
    'info.supersubs.South Africa',
    'info.supersubs.New Zealand',
    'info.outcome.eliminator',
    'info.outcome.result',
    'info.outcome.method',
    'info.neutral_venue',
    'info.match_type_number',
    'info.outcome.by.runs',
    'info.outcome.by.wickets'
], errors='ignore')
matches.shape

(4981, 7)

In [6]:
matches['info.gender'].value_counts()

info.gender
male      3130
female    1851
Name: count, dtype: int64

In [7]:
matches = matches[matches['info.gender'] == 'male'].copy()
matches = matches.drop(columns=['info.gender'])
matches

,innings,info.city,info.match_type,info.overs,info.teams,info.venue
0,"[{'1st innings': {'team': 'England', 'deliveri...",Bristol,T20,20,"[England, India]",County Ground
1,"[{'1st innings': {'team': 'India', 'deliveries...",Delhi,T20,20,"[India, New Zealand]",Arun Jaitley Stadium
2,"[{'1st innings': {'team': 'New Zealand', 'deli...",Rajkot,T20,20,"[New Zealand, India]",Saurashtra Cricket Association Stadium
3,"[{'1st innings': {'team': 'India', 'deliveries...",Thiruvananthapuram,T20,20,"[India, New Zealand]",Greenfield International Stadium
4,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",Abu Dhabi,T20,20,"[Sri Lanka, Pakistan]",Sheikh Zayed Stadium
...,...,...,...,...,...,...
4976,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",Mirpur,T20,20,"[Pakistan, Sri Lanka]",Shere Bangla National Stadium
4977,"[{'1st innings': {'team': 'Bangladesh', 'deliv...",Mirpur,T20,20,"[Bangladesh, India]",Shere Bangla National Stadium
4978,"[{'1st innings': {'team': 'Netherlands', 'deli...",Dubai,T20,20,"[United Arab Emirates, Netherlands]",ICC Academy
4979,"[{'1st innings': {'team': 'Australia', 'delive...",NaN,T20,20,"[Sri Lanka, Australia]",Pallekele International Cricket Stadium


In [8]:
matches['info.match_type'].value_counts()


info.match_type
T20    3130
Name: count, dtype: int64

In [9]:
matches['info.overs'].value_counts()

info.overs
20    3122
50       8
Name: count, dtype: int64

In [10]:
matches = matches[matches['info.overs'] == 20].copy()
matches = matches.drop(columns=['info.overs', 'info.match_type'])
matches

,innings,info.city,info.teams,info.venue
0,"[{'1st innings': {'team': 'England', 'deliveri...",Bristol,"[England, India]",County Ground
1,"[{'1st innings': {'team': 'India', 'deliveries...",Delhi,"[India, New Zealand]",Arun Jaitley Stadium
2,"[{'1st innings': {'team': 'New Zealand', 'deli...",Rajkot,"[New Zealand, India]",Saurashtra Cricket Association Stadium
3,"[{'1st innings': {'team': 'India', 'deliveries...",Thiruvananthapuram,"[India, New Zealand]",Greenfield International Stadium
4,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",Abu Dhabi,"[Sri Lanka, Pakistan]",Sheikh Zayed Stadium
...,...,...,...,...
4976,"[{'1st innings': {'team': 'Sri Lanka', 'delive...",Mirpur,"[Pakistan, Sri Lanka]",Shere Bangla National Stadium
4977,"[{'1st innings': {'team': 'Bangladesh', 'deliv...",Mirpur,"[Bangladesh, India]",Shere Bangla National Stadium
4978,"[{'1st innings': {'team': 'Netherlands', 'deli...",Dubai,"[United Arab Emirates, Netherlands]",ICC Academy
4979,"[{'1st innings': {'team': 'Australia', 'delive...",NaN,"[Sri Lanka, Australia]",Pallekele International Cricket Stadium


In [11]:
count = 1
delivery_dfs = []
for index, row in matches.iterrows():
    if count in [75, 108, 150, 180, 268, 360, 443, 458, 584, 748, 982, 1052, 1111, 1226, 1345]:
        count += 1
        continue
    
    ball_of_match = []
    batsman = []
    bowler = []
    runs = []
    player_of_dismissed = []
    teams_list = []
    batting_team = []
    match_id = []
    city = []
    venue = []
    
    try:
        innings_data = row['innings'][0]
        innings_key = '1st innings' if '1st innings' in innings_data else list(innings_data.keys())[0]
        
        for ball in innings_data[innings_key]['deliveries']:
            for key in ball.keys():
                match_id.append(count)
                batting_team.append(innings_data[innings_key]['team'])
                teams_list.append(row['info.teams'])
                ball_of_match.append(key)
                batsman.append(ball[key]['batsman'])
                bowler.append(ball[key]['bowler'])
                runs.append(ball[key]['runs']['total'])
                city.append(row['info.city'])
                venue.append(row['info.venue'])
                try:
                    player_of_dismissed.append(ball[key]['wicket']['player_out'])
                except:
                    player_of_dismissed.append('0')
                    
        loop_df = pd.DataFrame({
            'match_id': match_id,
            'teams': teams_list,
            'batting_team': batting_team,
            'ball': ball_of_match,
            'batsman': batsman,
            'bowler': bowler,
            'runs': runs,
            'player_dismissed': player_of_dismissed,
            'city': city,
            'venue': venue
        })
        delivery_dfs.append(loop_df)
    except Exception as e:
        print(f'Error processing match at index {index}: {e}')
        
    count += 1

delivery_df = pd.concat(delivery_dfs, ignore_index=True)


In [12]:
delivery_df   



,match_id,teams,batting_team,ball,batsman,bowler,runs,player_dismissed,city,venue
0,1,"[England, India]",England,0.1,JJ Roy,DL Chahar,1,0,Bristol,County Ground
1,1,"[England, India]",England,0.2,JC Buttler,DL Chahar,0,0,Bristol,County Ground
2,1,"[England, India]",England,0.3,JC Buttler,DL Chahar,4,0,Bristol,County Ground
3,1,"[England, India]",England,0.4,JC Buttler,DL Chahar,4,0,Bristol,County Ground
4,1,"[England, India]",England,0.5,JC Buttler,DL Chahar,0,0,Bristol,County Ground
...,...,...,...,...,...,...,...,...,...,...
375594,3122,"[Sri Lanka, Australia]",Sri Lanka,19.3,SMSM Senanayake,MA Starc,1,0,Colombo,R Premadasa Stadium
375595,3122,"[Sri Lanka, Australia]",Sri Lanka,19.4,DM de Silva,MA Starc,0,0,Colombo,R Premadasa Stadium
375596,3122,"[Sri Lanka, Australia]",Sri Lanka,19.5,DM de Silva,MA Starc,0,DM de Silva,Colombo,R Premadasa Stadium
375597,3122,"[Sri Lanka, Australia]",Sri Lanka,19.6,SMSM Senanayake,MA Starc,2,0,Colombo,R Premadasa Stadium


In [13]:
def bowl(row):
    for team in row['teams']:
        if team != row['batting_team']:
            return team

In [14]:

delivery_df['bowling_team'] = delivery_df.apply(bowl,axis=1)


In [15]:
delivery_df.drop(columns=['teams'],inplace=True)

In [16]:
delivery_df['batting_team'].unique()

array(['England', 'India', 'New Zealand', 'Sri Lanka', 'Pakistan',
       'Bangladesh', 'West Indies', 'Netherlands', 'Ireland', 'Scotland',
       'South Africa', 'Australia', 'Zimbabwe', 'United Arab Emirates',
       'Nepal', 'Oman', 'Papua New Guinea', 'Vanuatu', 'Philippines',
       'United States of America', 'Germany', 'Ghana', 'Uganda', 'Kenya',
       'Namibia', 'Nigeria', 'Botswana', 'Guernsey', 'Denmark', 'Jersey',
       'Italy', 'Norway', 'Thailand', 'Malaysia', 'Maldives', 'Singapore',
       'Kuwait', 'Bermuda', 'Canada', 'Cayman Islands', 'Hong Kong',
       'Portugal', 'Gibraltar', 'Spain', 'Bhutan', 'Qatar', 'Iran',
       'Belgium', 'Isle of Man', 'Bulgaria', 'Romania', 'Luxembourg',
       'Austria', 'Czech Republic', 'Greece', 'Serbia', 'Malta', 'Sweden',
       'Rwanda', 'Finland', 'Hungary', 'Estonia', 'Cyprus', 'Switzerland',
       'Seychelles', 'Malawi', 'Lesotho', 'Swaziland', 'Tanzania',
       'Mozambique', 'Sierra Leone', 'Cameroon', 'Bahrain', 'Belize',


In [17]:
teams = [
    'Australia',
    'India',
    'Bangladesh',
    'New Zealand',
    'South Africa',
    'England',
    'West Indies',
    'Afghanistan',
    'Pakistan',
    'Sri Lanka'    
]

In [18]:
delivery_df = delivery_df[delivery_df['batting_team'].isin(teams)]
delivery_df = delivery_df[delivery_df['bowling_team'].isin(teams)]


In [19]:
output = delivery_df[['match_id','batting_team','bowling_team','ball','runs','player_dismissed','city','venue']]

In [20]:
pickle.dump(output,open('dataset_level2.pkl','wb'))

In [21]:
output[output['city'].isnull()]['venue'].value_counts()

venue
Dubai International Cricket Stadium        2720
Pallekele International Cricket Stadium    2066
Melbourne Cricket Ground                   1201
Sydney Cricket Ground                       623
Adelaide Oval                               373
Harare Sports Club                          372
Sylhet International Cricket Stadium        128
Sharjah Cricket Stadium                     127
Carrara Oval                                 64
Name: count, dtype: int64

In [22]:
x = np.where(output['city'].isnull(), output['venue'].str.split().apply(lambda x:x[0]), output['city'])

In [23]:
output['city'] = x

C:\Users\Prakshil\AppData\Local\Temp\ipykernel_18280\2208188131.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  output['city'] = x


In [24]:
output.isnull().sum()

match_id            0
batting_team        0
bowling_team        0
ball                0
runs                0
player_dismissed    0
city                0
venue               0
dtype: int64

In [25]:
total_df = output.groupby('match_id').sum()['runs'].reset_index()
output = output.merge(total_df,on='match_id')

In [27]:
print(output['runs_x'].dtype)

int64


In [28]:
output['current_score'] = output.groupby('match_id')['runs_x'].cumsum()

In [29]:
output['over'] = output['ball'].apply(lambda x:str(x).split(".")[0])
output['ball_no'] = output['ball'].apply(lambda x:str(x).split(".")[1])

In [30]:
output['balls_bowled'] = (output['over'].astype('int')*6) + output['ball_no'].astype('int')

In [31]:
output['crr'] = round((output['current_score']*6)/output['balls_bowled'],2)

In [33]:
output['player_dismissed'] = output['player_dismissed'].apply(lambda x:0 if x=='0' else 1)
output['player_dismissed'] = output['player_dismissed'].astype('int')
output['player_dismissed'] = output.groupby('match_id')['player_dismissed'].cumsum()
output['wickets_left'] = 10 - output['player_dismissed']

In [34]:
final_df = output[['match_id','batting_team','bowling_team','runs_x','current_score','balls_bowled','wickets_left','crr','city','runs_y']]

In [35]:
final_df

,match_id,batting_team,bowling_team,runs_x,current_score,balls_bowled,wickets_left,crr,city,runs_y
0,1,England,India,1,1,1,9,6.00,Bristol,198
1,1,England,India,0,1,2,8,3.00,Bristol,198
2,1,England,India,4,5,3,7,10.00,Bristol,198
3,1,England,India,4,9,4,6,13.50,Bristol,198
4,1,England,India,0,9,5,5,10.80,Bristol,198
...,...,...,...,...,...,...,...,...,...,...
104624,3122,Sri Lanka,Australia,1,125,117,-112,6.41,Colombo,128
104625,3122,Sri Lanka,Australia,0,125,118,-113,6.36,Colombo,128
104626,3122,Sri Lanka,Australia,0,125,119,-114,6.30,Colombo,128
104627,3122,Sri Lanka,Australia,2,127,120,-115,6.35,Colombo,128


In [36]:
final_df = final_df.sample(final_df.shape[0])

In [37]:
final_df['balls_left'] = 120 - final_df['balls_bowled']
final_df['balls_left'] = final_df['balls_left'].apply(lambda x:0 if x<0 else x)

In [38]:
final_df['crr'] = round((final_df['current_score']*6)/final_df['balls_bowled'],2)

In [39]:
final_df


,match_id,batting_team,bowling_team,runs_x,current_score,balls_bowled,wickets_left,crr,city,runs_y,balls_left
85184,2864,West Indies,Sri Lanka,2,130,117,-110,6.67,Colombo,137,3
93145,2969,South Africa,New Zealand,1,51,50,-40,6.12,Chittagong,170,70
104336,3119,Bangladesh,India,4,58,48,-40,7.25,Mirpur,120,72
49209,1739,England,India,1,91,66,-57,8.27,Rajkot,171,54
24052,500,Australia,England,1,37,54,-45,4.11,Dubai,125,66
...,...,...,...,...,...,...,...,...,...,...,...
1950,19,Bangladesh,India,4,43,35,-26,7.37,Colombo,139,85
16814,363,India,England,1,60,37,-28,9.73,Ahmedabad,224,83
58996,2387,Sri Lanka,Pakistan,0,88,89,-81,5.93,Abu Dhabi,133,31
102221,3084,South Africa,West Indies,2,4,5,5,4.80,Nagpur,122,115


In [40]:
final_df.drop(columns=['balls_bowled'],inplace=True)

In [41]:
groups = final_df.groupby('match_id')

In [43]:
match_ids = final_df['match_id'].unique()
last_five = []

for id in match_ids:
    match_df = groups.get_group(id)
    
    last_five.extend(
        match_df['runs_x']   
        .rolling(window=30)
        .sum()
        .values
        .tolist()
    )

In [44]:
final_df['last_five'] = last_five


In [46]:
final_df

,match_id,batting_team,bowling_team,runs_x,current_score,wickets_left,crr,city,runs_y,balls_left,last_five
85184,2864,West Indies,Sri Lanka,2,130,-110,6.67,Colombo,137,3,NaN
93145,2969,South Africa,New Zealand,1,51,-40,6.12,Chittagong,170,70,NaN
104336,3119,Bangladesh,India,4,58,-40,7.25,Mirpur,120,72,NaN
49209,1739,England,India,1,91,-57,8.27,Rajkot,171,54,NaN
24052,500,Australia,England,1,37,-45,4.11,Dubai,125,66,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1950,19,Bangladesh,India,4,43,-26,7.37,Colombo,139,85,NaN
16814,363,India,England,1,60,-28,9.73,Ahmedabad,224,83,NaN
58996,2387,Sri Lanka,Pakistan,0,88,-81,5.93,Abu Dhabi,133,31,NaN
102221,3084,South Africa,West Indies,2,4,5,4.80,Nagpur,122,115,NaN


In [47]:
eligible_cities = final_df['city'].value_counts()[final_df['city'].value_counts() > 600].index.tolist()
final_df = final_df[final_df['city'].isin(eligible_cities)]


In [48]:
X = final_df.drop(columns=['match_id','runs_x','runs_y'])
y = final_df['runs_y']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [51]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error

trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'),
     ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

In [52]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',StandardScaler()),
    ('step3',XGBRegressor(n_estimators=1000,learning_rate=0.2,max_depth=12,random_state=1))
])

In [53]:
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.9255058842300837
4.331873414366629


In [ ]:
pickle.dump(output,open('cricket.pkl','wb'))